# Tool Chaining

### Setup

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv()

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

### Define Tools (Functions the Model Can Call)

In [2]:
# Tool 1: Summarize Text
def summarize_text(text: str) -> str:
    return f"SUMMARY: {text[:250]}..."

In [3]:
# Tool 2: Generate Quiz Questions
def generate_quiz(summary: str) -> str:
    return (
        "1. What is Agentic AI?\n"
        "2. Why is tool usage important for agents?\n"
        "3. Name one real-world use case of Agentic AI."
    )

### Register Tools with OpenAI (Function Calling Schema)

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "summarize_text",
            "description": "Summarize a given text",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {"type": "string"}
                },
                "required": ["text"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_quiz",
            "description": "Generate quiz questions from a summary",
            "parameters": {
                "type": "object",
                "properties": {
                    "summary": {"type": "string"}
                },
                "required": ["summary"]
            }
        }
    }
]

### User Prompt (Natural Language)

In [5]:
user_prompt = """
Summarize the following text and then create quiz questions from it.

Text:
Agentic AI systems can autonomously plan, reason, and act by using tools.
They are commonly used in workflows that require multi-step decision making.
"""

### First Model Call (Decides Which Tool to Use)

In [6]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": user_prompt}],
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message

### Execute Tool 1 (Summarization)

In [7]:
tool_call = message.tool_calls[0]
args = json.loads(tool_call.function.arguments)

# Call the LLM to actually summarize
summary_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Summarize the following text concisely."},
        {"role": "user", "content": args["text"]}
    ]
)

summary = summary_response.choices[0].message.content

print("🔹 Tool 1 Output (Summary):")
print(summary)

🔹 Tool 1 Output (Summary):
Agentic AI systems autonomously plan, reason, and act using tools, and are typically employed in multi-step decision-making workflows.


### Feed Output to Tool 2 (Tool Chaining)

In [8]:
quiz = generate_quiz(summary=summary)

print("\n🔹 Tool 2 Output (Final Result):")
print(quiz)


🔹 Tool 2 Output (Final Result):
1. What is Agentic AI?
2. Why is tool usage important for agents?
3. Name one real-world use case of Agentic AI.
